<a href="https://colab.research.google.com/github/programminghistorian/ph-submissions/blob/gh-pages/assets/visualisations-interactives-plotly/visualisations-interactives-plotly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Note aux utilisateur.ice.s :** Les figures qui s'affichent dans votre notebook adaptent leur taille à celle de la fenêtre de votre IDE ou navigateur. Le résultat peut être très insatisfaisant. Vous pouvez forcer la taille de la figure avec `fig.update_layout(height = xxx, width = xxx)`.

## Construire des visualisations avec Plotly Express

### Configurer Plotly Express

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

### Importer et nettoyer les données

In [ ]:
colonnes = ["Année","Origine sociale","Nombre de personnes admises",
    "Proportion de personnes admises","Type de baccalauréat"]

df : pd.DataFrame = pd.read_csv("data_article.csv",usecols = colonnes)

# On raccourcit le nom de chaque colonne
nouvelles_colonnes = {
    "Année" : "annee",
    "Origine sociale" : "origine_sociale",
    "Nombre de personnes admises" : "n_admis",
    "Proportion de personnes admises" : "p_admis",
    "Type de baccalauréat" : "type"
}
df.rename(nouvelles_colonnes, axis = 1, inplace = True)

# On raccourcit le nom de certaines origines sociales pour alléger les visualisations
df["origine_sociale"] = df["origine_sociale"].replace({
    "Professions intermédiaires" : "P. intermédiaires",
    "Cadres, professions intellectuelles supérieures" : "Cadres",
    "Autres personnes sans activité professionnelle" : "Sans activité p.",
    "Artisans, commerçants, chefs d'entreprise" : "Indépendants",
    "Agriculteurs exploitants" : "Agriculteurs",
})

# On raccourcit le nom des types de bacs
df["type"] = df["type"].replace({
    "baccalaureat général" : "bac_g",
    "baccalauréat technologique" : "bac_t",
    "baccalauréat professionnel" : "bac_p",
    "baccalauréat" : "bac_tous",
})

# On supprime les lignes associées aux origines sociales que l'on souhaite écarter 
# pour cette leçon.
df.drop(df[
        (df["origine_sociale"] == "Ensemble") | \
        (df["origine_sociale"] == "Indéterminé") 
    ].index, inplace = True)

### Diagrammes en barres

In [ ]:
# Création d'un nouveau DataFrame
num_admis_par_origine_sociale_2024 = df.loc[
    (df["annee"] == 2024)&(df["type"] == "bac_tous"), ["origine_sociale", "n_admis"]
]
print(num_admis_par_origine_sociale_2024)

In [ ]:
# Créer le diagramme en barres (bar chart) en utilisant la fonction .bar()
fig = px.bar(num_admis_par_origine_sociale_2024, x = "origine_sociale", y = "n_admis")

# Affiche la figure en utilisant la méthode .show()
fig.show()

**Figure 1.** Un diagramme en barres avec une interactivité simple en utilisant Plotly Express. Si les lecteur.ice.s survolent les barres, on peut y voir apparaître des boîtes flottantes.

In [ ]:
# Créer un diagramme en barres en utilisant la fonction .bar()
fig = px.bar(
    num_admis_par_origine_sociale_2024,
    x="origine_sociale",
    y="n_admis",
    title="Titre de votre choix",
    labels={"n_admis": "Nombre de personnes admises au baccalauréat"},

    # Notez que l'argument "color" prend une chaîne de caractères se référant à 
    # la colonne "origine_sociale" du jeu de données
    color="origine_sociale"
)

fig.show()

**Figure 2.** Un diagramme en barres avec une interactivité simple en utilisant Plotly Express. Cette visualisation est une variante de la Figure 1 avec cette fois-ci des couleurs et une légende interactive qui permet aux lecteur.ice.s d'isoler ou bien de retirer certaines barres.

### Courbes

In [ ]:
# Créer un nouveau DataFrame contenant le pourcentage d'admis au 
# baccalauréat (tous types confondus) par origine sociale et par année
prop_admis_par_origine_sociale_par_annee = df.loc[
    df["type"] == "bac_tous", ["annee", "origine_sociale", "p_admis"]
]

In [ ]:
# Créer des courbes avec la fonction px.line() et ajouter quelques éléments de 
# personnalisation
fig = px.line(
    prop_admis_par_origine_sociale_par_annee,
    x = "annee",
    y = "p_admis",
    # title = "Ajouter le titre de votre choix",
    labels = {"p_admis" : "Proportion de personnes admises au baccalauréat"},
    color = "origine_sociale"
)

fig.show()

**Figure 3.** Courbe avec une interactivité simple en utilisant Plotly Express. Survoler les lignes révèle une boîte flottante.

In [ ]:
fig.update_layout(
    font_family = "Courrier New",   # Modification de la police
    font_color = "blue",            # Modification de la couleur du texte
    legend_title_font_color = "red",# Modification de la couleur du titre de la légende
    title = "Un titre formaté"
)

fig.show()

**Figure 4.** Courbe avec une interactivité simple en utilisant Plotly Express. Survoler les lignes révèle une boîte flottante. Cette visualisation est une variante de la Figure 3 avec des polices d'écriture, couleurs et titres différents.

### Nuages de points

In [ ]:
num_admis_bac_t_bac_g = {
    "annee" : [], 
    "origine_sociale" : [],
    "n_admis_bac_g" : [],
    "n_admis_bac_t" : []
}

# Pour chaque combinaison d'année et origine sociale
for (annee, origine_sociale), _ in df.groupby(["annee", "origine_sociale"]):
    # On sauvegarde: l'année, l'origine sociale, le nombre de personnes admises 
    # au baccalauréat général et au baccalauréat technologique
    
    # Si l'origine sociale n'est pas dans la liste suivante, on passe à la prochaine.
    if origine_sociale not in ["Cadres", "Indépendants", "Ouvriers", "Agriculteurs"]:
        continue
    
    num_admis_bac_t_bac_g["annee"].append(annee)
    num_admis_bac_t_bac_g["origine_sociale"].append(origine_sociale)
    num_admis_bac_t_bac_g["n_admis_bac_g"].append(
        df.loc[
            (df["type"]=="bac_g")&\
            (df["annee"]==annee)&\
            (df["origine_sociale"]==origine_sociale),
            "n_admis"
        ].item()
    )
    num_admis_bac_t_bac_g["n_admis_bac_t"].append(
        df.loc[
            (df["type"]=="bac_t")&\
            (df["annee"]==annee)&\
            (df["origine_sociale"]==origine_sociale),
            "n_admis"
        ].item()
    )

# Transformons maintenant le dictionnaire en DataFrame.
num_admis_bac_t_bac_g = pd.DataFrame(num_admis_bac_t_bac_g)
print(num_admis_bac_t_bac_g)

In [ ]:
fig = px.scatter(
    num_admis_bac_t_bac_g,
    x="n_admis_bac_t",
    y="n_admis_bac_g",
    color = "origine_sociale"
    # title="Titre de votre choix",
)
fig.show()

**Figure 5.** Nuage de points avec une interactivité simple. Survoler un point du jeu de données permet d'afficher l'origine sociale ainsi que le nombre de personnes admises au baccalauréat général, puis le nombre de personnes admises au baccalauréat technologie (année non affichée). De plus, la légende interactive permet d'isoler, comparer, retirer des catégories de points.

## Créer une visualisation en mosaïque

In [ ]:
lignes_bac_technologique_et_general_2024 = \
    ((df["type"] == "bac_t") | (df["type"] == "bac_g")) &\
    (df["annee"] == 2024) # (type = "bac_t" ou "bac_g") ET annee = 2024
type_bac_origine_sociales = df.\
    loc[lignes_bac_technologique_et_general_2024, :]

# Utiliser la fonction px.bar pour spécifier le type de représentation
fig = px.bar(
    type_bac_origine_sociales,
    x="origine_sociale",
    y="n_admis",
    # Utiliser le paramètre facet_col pour spécifier la colonne qui doit subdiviser
    facet_col="type", 
    color="origine_sociale",
    # title="Titre de votre choix",
)
fig.show()

**Figure 6.** Une mosaïque de 2 diagrammes en barres avec une interactivité simple créée avec Plotly Express en distinguant le type de baccalauréat obtenu (technologique ou général). La légende interactive permet aussi d'isoler, comparer ou retirer certaines origines sociales.

### Ajouter des animations : évolution temporelle

In [ ]:
num_admis_par_origine_sociale_par_annee = df.loc[
    df["type"] == "bac_tous", ["annee", "origine_sociale", "n_admis"]
]
# On utilise px.bar pour créer un diagramme en barres
fig = px.bar(
    num_admis_par_origine_sociale_par_annee,
    x="origine_sociale",
    y="n_admis",
    labels={"n_admis": "Nombre de personnes admises au baccalauréat"},
    range_y=[0,200_000],  # Le paramètre range_y permet de personnaliser l'intervalle de l'axe y 
    color="origine_sociale",
    # title="Titre de votre choix",
    # Utiliser le paramètre animation_frame pour spécfier l'axe d'évolution
    animation_frame="annee", 
)
fig.show()

**Figure 7.** Diagramme en barres animé associé à une barre de défilement créé grâce à Plotly Express. Comme précédemment, les lecteur.ice.s peuvent survoler les barres pour faire apparaître des boîtes flottantes. Les lecteur.ice.s peuvent appuyer sur les boutons play/pause ou utiliser la barre de défilement pour naviguer à travers les années.

### Ajouter des animations : Menus déroulants

In [ ]:
fig = px.scatter(
    num_admis_bac_t_bac_g,
    x="n_admis_bac_t",
    y="n_admis_bac_g",
    color="origine_sociale", 
    # title="Titre de votre choix",
    labels = {
        "n_admis_bac_t" : "Nombre de personnes admises au baccalauréat technologique",
        "n_admis_bac_g" : "Nombre de personnes admises au baccalauréat général"
    }
)

In [ ]:
# Nous utilisons la méthode .update_layout pour ajouter le menu déroulant
fig.update_layout(
    updatemenus = [dict(
        buttons = [
            # Création de la liste de dictionaires pour chaque bouton du menu déroulant.
            dict(
                label = "Toutes les origines sociales", # Nom pour la première vue
                method = "update",
                args = [
                    # Cette vue montre les 4 origines sociales
                    {"visible" : [True, True, True, True]},
                    {
                        "title" : "Toutes les origines sociales",
                        "xaxis" : {
                            "title" : {
                                "text" : ("Nombre de personnes admises au"
                                              " baccalauréat technologique")
                            }
                        },
                        "yaxis" : {
                            "title" : {
                                "text" : ("Nombre de personnes admises au"
                                              " baccalauréat général")
                            }
                        }
                    }
                ]
            ),
            dict(
                label = "Agriculteurs", # Nom pour la deuxième vue
                method = "update",
                args = [
                    # Cette vue montre seulement la première origine sociale
                    {"visible" : [True, False, False, False]}, 
                    {
                        "title" : "Agriculteurs",
                        "xaxis" : {
                            "title" : {
                                "text" : ("Nombre de personnes admises au"
                                              " baccalauréat technologique")
                            }
                        },
                        "yaxis" : {
                            "title" : {
                                "text" : ("Nombre de personnes admises au"
                                              " baccalauréat général")
                            }
                        }
                    }
                ]
            ),
            dict(
                label = "Cadres", # Nom pour la troisième vue
                method = "update",
                args = [
                    # Cette vue montre uniquement la deuxième origine sociale
                    {"visible" : [False, True, False, False]}, 
                    {
                        "title" : "Cadres",
                        "xaxis" : {
                            "title" : {
                                "text" : ("Nombre de personnes admises au"
                                              " baccalauréat technologique")
                            }
                        },
                        "yaxis" : {
                            "title" : {
                                "text" : ("Nombre de personnes admises au"
                                              " baccalauréat général")
                            }
                        }
                    }
                ]
            ),
            dict(
                label = "Indépendants", # Nom pour la quatrième vue
                method = "update",
                args = [
                    # Cette vue montre la 3è origine sociale
                    {"visible" : [False, False, True, False]},
                    {
                        "title" : "Indépendants",
                        "xaxis" : {
                            "title" : {
                                "text" : ("Nombre de personnes admises au"
                                              " baccalauréat technologique")
                                }
                        },
                        "yaxis" : {
                            "title" : {
                                "text" : ("Nombre de personnes admises au"
                                              " baccalauréat général")
                            }
                        }
                    }
                ]
            ),
            dict(
                label = "Ouvriers", # Nom pour la cinquième vue
                method = "update",
                args = [
                    # Cette vue montre la 4è origine sociale
                    {"visible" : [False, False, False, True]},
                    {
                        "title" : "Ouvriers",
                        "xaxis" : {
                            "title" : {
                                "text" : ("Nombre de personnes admises au"
                                              " baccalauréat technologique")
                            }
                        },
                        "yaxis" : {
                            "title" : {
                                "text" : ("Nombre de personnes admises au"
                                              " baccalauréat général")
                            }
                        }
                    }
                ]
            ),
        ]
    )]
)

fig.show()

**Figure 8.** Nuage de points avec un filtre interactif sous la forme d'un menu déroulant créé grâce à Plotly Express. Cette figure contient une légende interactive qui permet au lecteur d'isoler, comparer et retirer des données. De plus survoler des points permet de faire apparaître des boîtes flottantes.

## Création des visualisations avec Plotly Graph Objects

### Configuration de Plotly Graph Objects

In [ ]:
import plotly.graph_objects as go 

> **Notons que dans un script `.py` conventionnel, les modules devraient être importés au début du script. On importe les modules ici pour un soucis de clarté.**

### Ce ne sont que des Objets ! La structure des données de Plotly Graph Objects 

In [ ]:
# Résultat du type de la figure
print(type(fig))

In [ ]:
print(fig.to_json(pretty = True)[0:500] + "\n...")

### Utiliser Plotly Graph Objects vs Plotly Express

In [ ]:
num_admis_par_origine_sociale_2024 = df.loc[
    (df["annee"] == 2024)&(df["type"] == "bac_tous"), ["origine_sociale", "n_admis"]
]

In [ ]:
fig = go.Figure(
    # Utiliser go.Bar() pour spécifier le type de représentation à créer
    go.Bar(
        x = num_admis_par_origine_sociale_2024["n_admis"], 
        y = num_admis_par_origine_sociale_2024["origine_sociale"],
        orientation = "h",
        # Nous devons formatter le "hover text" alors que c'est automatique avec plotly.px
        hovertemplate = ("Origine Sociale : %{y}<br>"
                         "Nombre de personnes admises : %{x}"
                         "<extra></extra>"  )
    ),
    # layout = {"title" : "Ajouter le titre de votre choix"},
)

fig.update_layout(
    # Nous devons modifier le layout pour les titres d'axes alors que c'est automatique avec plotly.px
    xaxis = {"title" : ("Nombre de personnes admises au baccalauréat (tous "
                        "types confondus)")},
    yaxis = {"title" : "Origine Sociale"}
)

fig.show()

**Figure 9.** Diagramme en barres horizontal avec une interactivité simple créé avec Plotly Graph Objects. Les lecteur.ices peuvent survoler les barres pour faire apparaître les boîtes flottantes.

In [ ]:
fig = px.bar(
    num_admis_par_origine_sociale_2024,
    x = "n_admis", y = "origine_sociale",
    orientation = "h",
    #title = "Titre de votre choix",
    labels = {"n_admis" : ("Nombre de personnes admises au baccalauréat (tous "
                           "types confondus)")}
)
fig.show()

**Figure 10.** Diagramme en barres horizontal avec une interactivité simple créé avec Plotly Express. Les lecteur.ices peuvent survoler les barres pour faire apparaître les boîtes flottantes.

### Pourquoi utiliser Plotly Graph Objects
#### Tableaux

In [ ]:
fig = go.Figure(
    data = [
        go.Table(
            header = {
                "values" : df.columns,
                "fill_color" : "paleturquoise",
                "align" : "left"
            },
            cells = {
                "values" : df.transpose().values.tolist(),
                "fill_color" : "lavender",
                "align" : "left"
            }
        )
    ]
)

fig.show()

**Figure 11.** Tableau contenant le jeu de données et créé avec Plotly Graph Object. Les Lecteur.ices peuvent faire défiler toutes les entrées du jeu de données comme iels le feraient dans un tableur.

#### La compositions de figures (*subplots*)

**Étape 1 : importer le module subplots et préparer les données**

In [ ]:
# Importer make_subplots
from plotly.subplots import make_subplots

# Préparation des données
num_admis_par_origine_sociale = df.\
    groupby(["type", "annee"]).\
    get_group(("bac_tous", 2024))

# On ne garde que 4 origines sociales pour alléger le graphe
selection_origines_sociales = np.isin(
    df["origine_sociale"], 
    ["Cadres", "Indépendants", "Ouvriers", "Agriculteurs"]
)
prop_admis_par_origine_sociale_par_annee = df.\
    loc[selection_origines_sociales, :].\
    groupby("type").get_group("bac_tous")

pourcentage_reussite_par_type_de_bac = df.\
    loc[df["type"] != "bac_tous",["type", "p_admis"]].\
    groupby("type")

**Étape 2 : Création d'une composition de sous-figures vide avec une grille 3x1 grâce à la fonction `make_subplots`**

In [ ]:
# 1 ligne, 3 colonnes
fig = make_subplots(rows = 1, cols = 3)

**Étape 3 : création de la première figure (le diagramme en barres) grace à la méthode `.add_trace()`**

In [ ]:
fig.add_trace(
    # Utiliser go.Bar() pour spécifier le type de représentation
    go.Bar(
        x = num_admis_par_origine_sociale["n_admis"],
        y = num_admis_par_origine_sociale["origine_sociale"],
        orientation = "h",
        name = "Nombre de personnes admises au baccalauréat",
        hovertemplate = ("<b>Origine sociale :</b> %{y}<br><b>Nombre de personnes "
                        "ayant obtenu le bac</b> : %{x}<extra></extra>")
    ),
    # Les paramètres row et col permettent de positionner la figure dans la bonne case
    row = 1, col = 1 
)

**Figure 12.** Une composition de figures avec 3 colonnes et une interactivité simple créée avec le module Plotly Graph Object, et avec un diagramme en barres sur la gauche montrant le nombre de personnes admises au baccalauréat (tous types confondus) par origine sociale en 2024, et deux colonnes vides sur la droite. Les lecteur.ices peuvent survoler les barres pour faire apparaître les boîtes flottantes.

> **Nota : si vous créez une composition de figures dans un Notebook Jupyter, relancer le code pourrait dupliquer la trace que vous venez d'ajouter et donc doubler la légende. Si vous avez besoin de relancer le code, il vaudrait mieux relancer à partir de la cellule qui définit la variable `fig` que vous modifiez.**

**Étape 4 : Ajouter la seconde figure (courbe)**

In [ ]:
# Pour chaque origine sociale il faut créer un objet go.Scatter différent afin de créer 
# les différentes courbes. 
# Pour se faire, on divise notre DataFrame par origine sociale et on procède comme 
# précédemment en ne travaillant qu'avec les sous-dataset
for origine_sociale, df_origine_sociale in prop_admis_par_origine_sociale_par_annee.\
                                    groupby("origine_sociale") :
    fig.add_trace(
        # Utiliser go.Scatter() pour spécifier le type de représentation
        go.Scatter(
            x = df_origine_sociale["annee"],
            y = df_origine_sociale["p_admis"],
            name = origine_sociale,
            mode = "markers+lines",
            hovertemplate = (f"<b>Origine sociale :</b> {origine_sociale}<br>"
                             "<b>Année :</b> %{x}<br>"
                             "<b>Proportion de personnes admises :</b> %{y}")  

        ),
        # Les paramètres row et col permettent de positionner la figure dans la bonne case
        row = 1, col = 2 
    )
    
fig.show()

**Figure 13.** Une composition de figures avec 3 colonnes et une interactivité simple créée avec le module Plotly Graph Object, et avec un diagramme en barres sur la gauche montrant le nombre de personnes admises au baccalauréat (tous types confondus) par origine sociale en 2024, une courbe au centre montrant l'évolution de la proportion de personnes admises au baccalauréat (tous types confondus) par origine sociale et une colonnes vide sur la droite. Les lecteur.ices peuvent survoler les barres pour faire apparaître les boîtes flottantes.

**Étape 5 : Ajouter la dernière figure (diagramme en boîte)**

In [ ]:
fig.add_trace(
    # Utiliser go.Box() pour spécifier le type de représentation
    go.Box(
        y = pourcentage_reussite_par_type_de_bac.\
            get_group("bac_g")["p_admis"],
        name = "Baccalauréat général"),
        row = 1, col = 3 # puisque c'est la troisième, on le met sur la 3è colonne
)

# On ajoute le deuxième et troisiéme diagramme en boîte puisqu'on a 3 groupes 
# distincts pour chaque type de baccalauréat
fig.add_trace(
    go.Box(
        y = pourcentage_reussite_par_type_de_bac.\
            get_group("bac_t")["p_admis"],
        name = "Baccalauréat technologique"),
        row = 1, col = 3 # puisque c'est la troisième, on le met sur la 3è colonne
)

fig.add_trace(
    go.Box(
        y = pourcentage_reussite_par_type_de_bac.\
            get_group("bac_p")["p_admis"],
        name = "Baccalauréat professionnel"),
        row = 1, col = 3 # puisque c'est la troisième, on le met sur la 3è colonne
)

**Figure 14.** Une composition de figures avec 3 colonnes et une interactivité simple créée avec le module Plotly Graph Object, et avec un diagramme en barres sur la gauche montrant le nombre de personnes admises au baccalauréat (tous types confondus) par origine sociale en 2024, une courbe au centre montrant l'évolution de la proportion de personnes admises au baccalauréat (tous types confondus) par origine sociale et trois diagrammes en boîte représentant la distribution de la part de personnes admises selon le type de baccalauréat. Les lecteur.ices peuvent survoler les barres pour faire apparaître les boîtes flottantes.

**Étape 6 : Formattage de la Figure**

In [ ]:
fig.update_layout(
    # Changement de la police d'écriture pour toute la figure
    font_family = "Times New Roman", 
    # Changement de la police d'écriture pour les notes hover
    hoverlabel_font_family = "Times New Roman", 
    # changement de la taille d'écriture pour les notes hover
    hoverlabel_font_size = 16, 
    # title_text = "Ajouter un titre ici", # Titre principal
    # Positionnement du titre principal au centre de la visualisation 
    # (note : le paramètre title_x ne prend que des entiers (integers) 
    # ou des réels (floats))
    # title_x = 0.5 
    # ajout d'un titre d'axe pour l'absisse de la première figure
    xaxis1_title_text = ("Nombre de personnes admises au baccalauréat <br>(tous "
                         "types confondus) en 2024"),
    # ajout d'un titre d'axe pour les ordonnées de la première figure
    yaxis1_title_text = "Origine sociale", 
    yaxis2_title_text = ("Part de personnes admises au baccalauréat (tous types"
                         " confondus)"),
    xaxis2_title_text = "Année",
    yaxis3_title_text = "Distribution du pourcentage d'admission au baccalauréat",
    showlegend = False, # Retire la légende
    # Ajuste la taille de la visualisation  - pas toujours nécessaire mais peut 
    # s'avérer utile si les figures sont publiées sur internet
    height = 650
)

**Figure 15.** Une composition de figures avec 3 colonnes et une interactivité simple créée avec le module Plotly Graph Object, et avec un diagramme en barres sur la gauche montrant le nombre de personnes admises au baccalauréat (tous types confondus) par origine sociale en 2024, une courbe au centre montrant l'évolution de la proportion de personnes admises au baccalauréat (tous types confondus) par origine sociale et trois diagrammes en boîte représentant la distribution de la part de personnes admises selon le type de baccalauréat. Les lecteur.ices peuvent survoler les barres pour faire apparaître les boîtes flottantes. Cette visualisation est une variante de la Figure 14 avec une personalisation avancée.

**Étape 7 : Ajout d'annotations aux courbes**

In [ ]:
fig.update_layout(
    # annotations reçoit une liste de disctionnaires, un dictionnaire = une annotation
    annotations = [
        # Notre première annotation sera pour identifier l'origine sociale "Agriculteurs"
        dict(
            # coordonnées du points de référence de l'annotation
            x = 2000, y = 85,
            # Spécifie dans quel référentiel on se place, ici comme on annote la
            # figure n°2 on donne comme référence x2, y2
            xref = "x2", yref = "y2",
            # Permet de spécifier la longueur de la flèche, et donc le décalage du 
            # texte par rapport au point
            ax = 0, ay = -100,
            text = "Agriculteurs",
            showarrow = True, # Utilisez False si vous ne voulez pas de la tête
            # de flèche dans l'annotation
            arrowhead = 1, # change la taille de la tête de flèche
        ),
        # Notre deuxième annotation sera pour identifier l'origine sociale "Indépendants"
        dict(
            x = 2001, y = 78.99,
            xref = "x2", yref = "y2",ax = 130, ay = 30,
            text = "Indépendants",showarrow = True, arrowhead = 1, 
        ),
        # Notre troisième annotation sera pour identifier l'origine sociale "Ouvriers"
        dict(
            x = 2019, y = 85.40,
            xref = "x2", yref = "y2",ax = 10, ay = 50,
            text = "Ouvriers",showarrow = True, arrowhead = 1, 
        ),
        # Notre quatrième annotation sera pour identifier l'origine sociale "Cadres"
        dict(
            x = 2020, y = 98.25,
            xref = "x2", yref = "y2",ax = -100, ay = 0,
            text = "Cadres",showarrow = True, arrowhead = 1, 
        ),
    ]
)

**Figure 16.** Une composition de figures avec 3 colonnes et une interactivité simple créée avec le module Plotly Graph Object, et avec un diagramme en barres sur la gauche montrant le nombre de personnes admises au baccalauréat (tous types confondus) par origine sociale en 2024, une courbe au centre montrant l'évolution de la proportion de personnes admises au baccalauréat (tous types confondus) par origine sociale et trois diagrammes en boîte représentant la distribution de la part de personnes admises selon le type de baccalauréat. Les lecteur.ices peuvent survoler les barres pour faire apparaître les boîtes flottantes. Cette visualisation est une variante de la Figure 15 des annotations pour repérer les courbes de la sous-figure du milieu.

**Étape 8 : Ajout d'annotations en dessous de la figure**

In [ ]:
fig.add_annotation(
    dict(
        font=dict(color="black", size=15),  # Change la police d'écriture
        x=0.5,  # Utilise x et y pour la position de l'annotation
        y=-0.4,
        showarrow=False,
        text=(
            "Nombre de personnes admises au baccalauréat (tous types confondus)"
            " en 2024 et par origine sociale (gauche); <br>"
            "Proportion de personnes admises au baccalauréat (tous types"
            " confondus) à travers les années et par origine sociale (centre);<br>"
            "Distribution du pourcentage d'admission pour le baccalauréat général,"
            " technologique et professionnel (droite)."),
        # Option pour changer l'orientation de l'écriture, utile pour la gestion de l'espace
        textangle=0,  
        xanchor="center",
        # Régler xref et yref à 'paper' pour que les valeurs de x et y soient 
        # des coordonées absolues
        xref="paper",  
        yref="paper",
    )
)
# On ajoute une marge pour que laisser la place aux annotations
fig.update_layout(margin = {"b" : 200})

**Figure 17.** Une composition de figures avec 3 colonnes et une interactivité simple créée avec le module Plotly Graph Object, et avec un diagramme en barres sur la gauche montrant le nombre de personnes admises au baccalauréat (tous types confondus) par origine sociale en 2024, une courbe au centre montrant l'évolution de la proportion de personnes admises au baccalauréat (tous types confondus) par origine sociale et trois diagrammes en boîte représentant la distribution de la part de personnes admises selon le type de baccalauréat. Cette visualisation est une variante de la Figure 16 avec des annotations supplémentaires sous les figures.

## Afficher et Exporter les visualisations

In [ ]:
fig = px.line(
    prop_admis_par_origine_sociale_par_annee,
    x = "annee",
    y = "p_admis",
    # title = "Ajouter le titre de votre choix",
    labels = {"p_admis" : "Proportion de personnes admises au baccalauréat"},
    color = "origine_sociale"
)

### Afficher la visualisation

In [ ]:
fig.show()

**Figure 18.** Reproduction de la Figure 3, illustrant la fonction fig.show().

### Export des visualisations

#### Export en HTML 

In [ ]:
# Sauvegarde de la visualisation en format HTML
fig.write_html("nom_visualisation.html")

#### Export d'images statistiques

In [ ]:
# Export en images classiques (raster ):
fig.write_image("nom_visualisation.png")
fig.write_image("nom_visualisation.jpeg")

# Export en images vectorielles :
fig.write_image("nom_visualisation.svg")
fig.write_image("nom_visualisation.pdf")